# Experimento: 3 layouts × 4 estratégias × 100 pedidos

Cada um dos 100 pedidos de avaliação é coletado nos layouts L0, L1 e L2 com as quatro estratégias de `rotas.ipynb`, num total de 1.200 rotas. As tabelas são salvas em `resultados/` e usadas no relatório e em `figuras/figuras.ipynb`.

In [1]:
%run grafo.ipynb
%run rotas.ipynb

from itertools import combinations, product

CARRINHO = "ordem_carrinho"
COMPARADAS = [e for e in ESTRATEGIAS if e != CARRINHO]
COMPARACOES = [("L0", "L1"), ("L0", "L2"), ("L2", "L1")]
FAIXAS_SECOES = [("1-2", 1, 2), ("3-4", 3, 4), ("5-6", 5, 6), ("7-8", 7, 8),
                 ("9-11", 9, 11), ("12-15", 12, 15), ("16+", 16, np.inf)]
EMPATE = 1e-9

PASTA_RESULTADOS.mkdir(exist_ok=True)

## Medidas

- **Ganho** de uma estratégia: quanto ela anda a menos que a ordem do carrinho no mesmo pedido.
- **Redução** entre layouts: quanto a distância média cai do layout base para o novo.
- **Pedidos por turno:** quantos pedidos cabem em 8 horas com o tempo médio de coleta.
- **IC de 95%** por bootstrap pareado (10.000 reamostragens).

In [2]:
def ganho(carrinho, metros):
    return (carrinho - metros) / carrinho


def ganho_medio(carrinho, metros):
    return float(np.mean(ganho(carrinho, metros)))


def reducao(base, novo):
    return 1 - np.mean(novo) / np.mean(base)


def pedidos_por_turno(tempos):
    return DURACAO_TURNO / np.mean(tempos)


def bootstrap(medida, *amostras, repeticoes=10000, semente=42):
    # bootstrap pareado: o mesmo sorteio de pedidos vale para todas as amostras
    amostras = [np.asarray(a, dtype=float) for a in amostras]
    n = len(amostras[0])
    sorteios = np.random.default_rng(semente).integers(0, n, size=(repeticoes, n))
    valores = [medida(*(a[sorteio] for a in amostras)) for sorteio in sorteios]
    inferior, superior = np.percentile(valores, [2.5, 97.5])
    return float(inferior), float(superior)


def intervalo(sufixo, escala, medida, *amostras):
    inferior, superior = bootstrap(medida, *amostras)
    return {f"ic_inf_{sufixo}": escala * inferior, f"ic_sup_{sufixo}": escala * superior}


def salvar(tabela, nome):
    tabela.to_csv(PASTA_RESULTADOS / f"{nome}.csv", index=False, float_format="%.3f")
    return tabela

## Rotas

Para cada pedido, as seções viram slots do layout e cada estratégia gera uma rota ENTRADA → slots → CAIXA, medida pela matriz de distâncias do Dijkstra.

In [3]:
linhas = []
for layout in LAYOUTS:
    posicoes = carregar_layout(layout)
    for pedido in pedidos.itertuples():
        slots = slots_do_pedido(pedido.secoes, posicoes, loja)
        metros = {e: executar(e, slots, distancias, loja)[1] for e in ESTRATEGIAS}
        for estrategia in ESTRATEGIAS:
            linhas.append({
                "id_teste": pedido.id_teste,
                "order_id": pedido.order_id,
                "layout": layout,
                "estrategia": estrategia,
                "n_itens": pedido.n_itens,
                "n_secoes": len(slots),
                "n_corredores": len(set(loja.corredor[slots])),
                "pedido_heuristico": not usa_held_karp(slots),
                "distancia_m": metros[estrategia],
                "tempo_s": tempo_coleta(metros[estrategia], pedido.n_itens),
                "ganho_vs_carrinho": ganho(metros[CARRINHO], metros[estrategia]),
            })

resultados = pd.DataFrame(linhas)
resultados.to_csv(PASTA_RESULTADOS / "resultados_por_pedido.csv", index=False)
print(len(resultados), "rotas avaliadas")
resultados.head()

1200 rotas avaliadas


,id_teste,order_id,layout,estrategia,n_itens,n_secoes,n_corredores,pedido_heuristico,distancia_m,tempo_s,ganho_vs_carrinho
0,1,21461,L0,ordem_carrinho,12,3,3,False,111.5,411.5,0.0
1,1,21461,L0,corredores,12,3,3,False,111.5,411.5,0.0
2,1,21461,L0,vizinho,12,3,3,False,111.5,411.5,0.0
3,1,21461,L0,custoMin,12,3,3,False,111.5,411.5,0.0
4,2,40098,L0,ordem_carrinho,8,7,5,False,365.5,605.5,0.0


Para as tabelas, os resultados ficam com uma linha por pedido e uma coluna por (layout, estratégia).

In [4]:
metros = resultados.pivot(index="id_teste", columns=["layout", "estrategia"], values="distancia_m")
tempos = resultados.pivot(index="id_teste", columns=["layout", "estrategia"], values="tempo_s")

carrinho = resultados[resultados["estrategia"] == CARRINHO]
secoes = carrinho.pivot(index="id_teste", columns="layout", values="n_secoes")
corredores = carrinho.pivot(index="id_teste", columns="layout", values="n_corredores")
print("Corredores distintos por pedido (média):", corredores.mean().round(2).to_dict())
heuristico = (carrinho[carrinho["layout"] == "L0"].set_index("id_teste")["pedido_heuristico"]
              .sort_index().to_numpy())

Corredores distintos por pedido (média): {'L0': 5.85, 'L1': 5.27, 'L2': 6.14}


Dois números citados no relatório: a posição média das seções ponderada pela demanda (1 = frente da loja) e em quantos pedidos `Pedidos.csv` lista as seções fora da ordem do carrinho.

In [5]:
demanda = pd.read_csv(PASTA_DADOS / "curados" / "frequencia_aisles.csv").set_index("aisle_id")["pedidos_prior"]
for layout in LAYOUTS:
    posicoes = carregar_layout(layout)
    media = np.average([posicoes[s][1] for s in demanda.index], weights=demanda)
    print(f"Posição média ponderada pela demanda em {layout}: {media:.2f}")

nomes = nomes_das_secoes()
listadas = pd.read_csv(PASTA_DADOS / "amostras" / "Pedidos.csv", encoding="utf-8-sig")
listadas = listadas.set_index("order_id")["seções do pedido"].str.split(";").map(lambda s: [n.strip() for n in s])
no_carrinho = pedidos.set_index("order_id")["secoes"].map(lambda s: [nomes[a] for a in s])
fora_de_ordem = sum(listadas[o] != no_carrinho[o] for o in no_carrinho.index)
print("Pedidos com as seções fora da ordem do carrinho em Pedidos.csv:", fora_de_ordem)

Posição média ponderada pela demanda em L0: 4.50
Posição média ponderada pela demanda em L1: 3.97
Posição média ponderada pela demanda em L2: 2.15
Pedidos com as seções fora da ordem do carrinho em Pedidos.csv: 82


## Tabelas

### Distância por layout e estratégia

Média e mediana da distância dos 100 pedidos em cada combinação, com o IC de 95% da média.

In [6]:
linhas = []
for layout, estrategia in product(LAYOUTS, ESTRATEGIAS):
    m = metros[layout, estrategia]
    linhas.append({"layout": layout, "estrategia": estrategia, "media_m": m.mean(),
                   "mediana_m": m.median(), **intervalo("m", 1, np.mean, m)})

salvar(pd.DataFrame(linhas), "tabela_distancia")

,layout,estrategia,media_m,mediana_m,ic_inf_m,ic_sup_m
0,L0,ordem_carrinho,313.405,277.25,279.644125,346.873750
1,L0,corredores,160.175,159.00,150.125000,170.050000
2,L0,vizinho,165.295,154.25,153.584875,177.205125
3,L0,custoMin,155.550,153.00,146.154875,164.760000
4,L1,ordem_carrinho,334.330,297.25,296.819625,372.106625
5,L1,corredores,147.525,142.75,138.950000,156.175000
6,L1,vizinho,151.435,141.50,141.670000,161.425375
7,L1,custoMin,144.115,139.00,136.175000,152.165000
8,L2,ordem_carrinho,263.870,230.75,233.829875,295.110250
9,L2,corredores,118.850,111.50,112.225000,125.750000


### Redução de distância entre layouts

Cada estratégia é comparada em L1 contra L0, L2 contra L0 e L1 contra L2. A redução é $1 - \bar{D}_{novo} / \bar{D}_{base}$: valor negativo quer dizer que o layout novo andou mais.

In [7]:
linhas = []
for estrategia, (base, novo) in product(ESTRATEGIAS, COMPARACOES):
    antes, depois = metros[base, estrategia], metros[novo, estrategia]
    linhas.append({"estrategia": estrategia, "comparacao": f"{novo} vs {base}",
                   "media_base_m": antes.mean(), "media_novo_m": depois.mean(),
                   "reducao_pct": 100 * reducao(antes, depois),
                   **intervalo("pct", 100, reducao, antes, depois)})

reducoes = salvar(pd.DataFrame(linhas), "tabela_reducao_layouts")
reducoes

,estrategia,comparacao,media_base_m,media_novo_m,reducao_pct,ic_inf_pct,ic_sup_pct
0,ordem_carrinho,L1 vs L0,313.405,334.330,-6.676664,-12.163533,-1.368870
1,ordem_carrinho,L2 vs L0,313.405,263.870,15.805427,10.755264,20.690734
2,ordem_carrinho,L1 vs L2,263.870,334.330,-26.702543,-35.351830,-18.635180
3,corredores,L1 vs L0,160.175,147.525,7.897612,5.153894,10.616222
4,corredores,L2 vs L0,160.175,118.850,25.799906,23.180046,28.329684
5,corredores,L1 vs L2,118.850,147.525,-24.127051,-28.085969,-20.233960
6,vizinho,L1 vs L0,165.295,151.435,8.385009,4.754512,12.068884
7,vizinho,L2 vs L0,165.295,135.360,18.110046,13.372570,22.571809
8,vizinho,L1 vs L2,135.360,151.435,-11.875739,-18.293879,-5.646465
9,custoMin,L1 vs L0,155.550,144.115,7.351334,4.789546,9.895002


### Ganho contra a ordem do carrinho

O ganho é calculado pedido a pedido e depois tirada a média, para que pedidos grandes não pesem mais que os pequenos. A tabela também registra quantos pedidos usaram a rota heurística (mais de 15 seções).

In [8]:
linhas = []
for layout, estrategia in product(LAYOUTS, COMPARADAS):
    base, m = metros[layout, CARRINHO], metros[layout, estrategia]
    linhas.append({"layout": layout, "estrategia": estrategia,
                   "ganho_medio_pct": 100 * ganho_medio(base, m), **intervalo("pct", 100, ganho_medio, base, m),
                   "ganho_mediano_pct": 100 * ganho(base, m).median(),
                   "n_pedidos_heuristicos": int(heuristico.sum())})

ganhos = salvar(pd.DataFrame(linhas), "tabela_ganho")
ganhos

,layout,estrategia,ganho_medio_pct,ic_inf_pct,ic_sup_pct,ganho_mediano_pct,n_pedidos_heuristicos
0,L0,corredores,39.370566,35.212136,43.225850,45.426794,8
1,L0,vizinho,38.228667,34.054855,42.152228,44.049722,8
2,L0,custoMin,40.658983,36.427572,44.618030,47.003249,8
3,L1,corredores,44.913611,40.189343,49.300339,52.649370,8
4,L1,vizinho,44.153866,39.463494,48.510367,51.937002,8
5,L1,custoMin,45.784028,41.049758,50.220087,53.601319,8
6,L2,corredores,43.439551,38.762913,48.034444,52.728258,8
7,L2,vizinho,38.645364,33.961619,43.225762,47.120628,8
8,L2,custoMin,43.591794,38.930436,48.182068,52.728258,8


### Ganho por faixa de seções

Os pedidos são agrupados pelo número de seções distintas, que muda pouco entre os layouts.

In [9]:
def ganho_por_faixa(contagem, faixas, coluna):
    linhas = []
    for layout, (nome, minimo, maximo) in product(LAYOUTS, faixas):
        na_faixa = contagem[layout].between(minimo, maximo).to_numpy()
        if not na_faixa.any():
            continue
        base = metros[layout, CARRINHO][na_faixa]
        for estrategia in COMPARADAS:
            m = metros[layout, estrategia][na_faixa]
            linhas.append({"layout": layout, coluna: nome, "n_pedidos": int(na_faixa.sum()),
                           "estrategia": estrategia, "ganho_medio_pct": 100 * ganho_medio(base, m),
                           **intervalo("pct", 100, ganho_medio, base, m),
                           "n_pedidos_heuristicos": int(heuristico[na_faixa].sum())})
    return pd.DataFrame(linhas)


faixas = salvar(ganho_por_faixa(secoes, FAIXAS_SECOES, "faixa_secoes"), "tabela_ganho_faixa_secoes")
faixas[faixas["layout"] == "L0"]

,layout,faixa_secoes,n_pedidos,estrategia,ganho_medio_pct,ic_inf_pct,ic_sup_pct,n_pedidos_heuristicos
0,L0,1-2,14,corredores,3.361345,0.000000,10.084034,0
1,L0,1-2,14,vizinho,3.361345,0.000000,10.084034,0
2,L0,1-2,14,custoMin,3.361345,0.000000,10.084034,0
3,L0,3-4,17,corredores,28.378399,21.176169,35.543017,0
4,L0,3-4,17,vizinho,27.806679,20.002866,35.443691,0
5,L0,3-4,17,custoMin,28.801083,21.706090,35.817894,0
6,L0,5-6,18,corredores,39.188500,34.728347,43.676417,0
7,L0,5-6,18,vizinho,40.608100,35.664507,45.497969,0
8,L0,5-6,18,custoMin,41.496394,37.054392,45.942790,0
9,L0,7-8,15,corredores,46.638690,40.480985,51.665838,0


### Tempo de coleta e pedidos por turno

O tempo de cada pedido é $T = D/v + t_{item} \cdot itens + t_{fixo}$, e os pedidos por turno são 8 h divididas pelo tempo médio. Como a parte fixa não depende da rota, o ganho em tempo é menor que o ganho em distância.

In [10]:
linhas = []
for layout, estrategia in product(LAYOUTS, ESTRATEGIAS):
    t = tempos[layout, estrategia]
    linhas.append({"layout": layout, "estrategia": estrategia, "tempo_medio_s": t.mean(),
                   **intervalo("s", 1, np.mean, t), "pedidos_turno": pedidos_por_turno(t),
                   **intervalo("turno", 1, pedidos_por_turno, t)})

salvar(pd.DataFrame(linhas), "tabela_tempo_turno")

,layout,estrategia,tempo_medio_s,ic_inf_s,ic_sup_s,pedidos_turno,ic_inf_turno,ic_sup_turno
0,L0,ordem_carrinho,595.255,541.204500,648.721375,48.382626,44.395022,53.214635
1,L0,corredores,442.025,411.100000,472.851250,65.154686,60.907103,70.055947
2,L0,vizinho,447.145,414.674875,479.911000,64.408637,60.011127,69.452001
3,L0,custoMin,437.400,406.829375,467.885250,65.843621,61.553554,70.791348
4,L1,ordem_carrinho,616.180,559.744500,673.295125,46.739589,42.774704,51.452046
5,L1,corredores,429.375,400.099375,458.925000,67.074236,62.755352,71.982117
6,L1,vizinho,433.285,403.103000,464.095125,66.468952,62.056243,71.445760
7,L1,custoMin,425.965,397.238000,455.066125,67.611189,63.287506,72.500617
8,L2,ordem_carrinho,545.720,495.514250,597.225875,52.774316,48.222961,58.121436
9,L2,corredores,400.700,373.275000,428.676875,71.874220,67.183470,77.154913


### Análise de sensibilidade

Os parâmetros de tempo são hipóteses do grupo, então os pedidos por turno são recalculados para $v \in \{0{,}8;\ 1{,}0;\ 1{,}1\}$ m/s, $t_{\text{item}} \in \{10, 15, 20\}$ s e $t_{\text{fixo}} \in \{60, 120, 180\}$ s, sem recalcular as rotas. A tabela mostra a faixa de pedidos por turno de cada caso e se a ordem dos doze casos se mantém em todas as 27 combinações.

In [11]:
VELOCIDADES, TEMPOS_ITEM, TEMPOS_FIXOS = [0.8, 1.0, 1.1], [10, 15, 20], [60, 120, 180]
itens = carrinho[carrinho["layout"] == "L0"].set_index("id_teste")["n_itens"].sort_index()

turnos = {}
for v, t_item, t_fixo in product(VELOCIDADES, TEMPOS_ITEM, TEMPOS_FIXOS):
    tempo = metros.div(v).add(t_item * itens + t_fixo, axis=0)
    turnos[v, t_item, t_fixo] = DURACAO_TURNO / tempo.mean()
turnos = pd.DataFrame(turnos)

ordem_base = turnos[VELOCIDADE, SEGUNDOS_POR_ITEM, SEGUNDOS_POR_PEDIDO].rank().to_numpy()
ordem_mantida = all((turnos[c].rank().to_numpy() == ordem_base).all() for c in turnos)

sensibilidade = pd.DataFrame({"pedidos_turno": turnos[VELOCIDADE, SEGUNDOS_POR_ITEM, SEGUNDOS_POR_PEDIDO],
                              "minimo": turnos.min(axis=1), "maximo": turnos.max(axis=1)})
sensibilidade = salvar(sensibilidade.reset_index(), "tabela_sensibilidade")
print("Ordem dos 12 casos mantida em todas as combinações:", ordem_mantida)
sensibilidade

Ordem dos 12 casos mantida em todas as combinações: True


,layout,estrategia,pedidos_turno,minimo,maximo
0,L0,ordem_carrinho,48.382626,36.568817,63.602325
1,L0,corredores,65.154686,48.320627,91.862033
2,L0,vizinho,64.408637,47.807277,90.518165
3,L0,custoMin,65.843621,48.793918,93.110745
4,L1,ordem_carrinho,46.739589,35.393336,61.038110
5,L1,corredores,67.074236,49.637521,95.359933
6,L1,vizinho,66.468952,49.222881,94.250651
7,L1,custoMin,67.611189,50.004883,96.348900
8,L2,ordem_carrinho,52.774316,39.689239,70.626003
9,L2,corredores,71.874220,52.905922,104.368452


### Estratégias equivalentes

Pedidos em que duas estratégias (ou as quatro) andam exatamente a mesma distância.

In [12]:
duas_ou_mais = (secoes["L0"] >= 2).to_numpy()
linhas = []
for layout in LAYOUTS:
    casos = {f"{a} = {b}": ((metros[layout, a] - metros[layout, b]).abs() <= EMPATE).to_numpy()
             for a, b in combinations(ESTRATEGIAS, 2)}
    pares = list(casos.values())
    casos["qualquer par"] = np.any(pares, axis=0)
    casos["as 4 iguais"] = np.all(pares, axis=0)
    for caso, empatou in casos.items():
        linhas.append({"layout": layout, "caso": caso, "n_pedidos": int(empatou.sum()),
                       "pct_pedidos": 100 * empatou.mean(),
                       "n_pedidos_2mais_secoes": int((empatou & duas_ou_mais).sum()),
                       "pct_pedidos_2mais_secoes": 100 * (empatou & duas_ou_mais).sum() / duas_ou_mais.sum()})

salvar(pd.DataFrame(linhas), "tabela_equivalencias")

,layout,caso,n_pedidos,pct_pedidos,n_pedidos_2mais_secoes,pct_pedidos_2mais_secoes
0,L0,ordem_carrinho = corredores,14,14.0,5,5.494505
1,L0,ordem_carrinho = vizinho,15,15.0,6,6.593407
2,L0,ordem_carrinho = custoMin,14,14.0,5,5.494505
3,L0,corredores = vizinho,40,40.0,31,34.065934
4,L0,corredores = custoMin,55,55.0,46,50.549451
5,L0,vizinho = custoMin,60,60.0,51,56.043956
6,L0,qualquer par,79,79.0,70,76.923077
7,L0,as 4 iguais,14,14.0,5,5.494505
8,L1,ordem_carrinho = corredores,14,14.0,5,5.494505
9,L1,ordem_carrinho = vizinho,15,15.0,6,6.593407


## Hipóteses

- **H1:** L1 e L2 andam menos que L0.
- **H2:** L1 anda menos que L2.
- **H3:** as estratégias ganham da ordem do carrinho, `custoMin` nunca perde para as outras e o ganho cresce com o número de seções.
- **H4:** com uma seção só, as quatro estratégias empatam.

Pelo intervalo, a hipótese é confirmada quando o IC fica todo acima de zero e contrariada quando fica todo abaixo. Pela contagem, é confirmada quando nenhum pedido a viola.

In [13]:
def pelo_intervalo(inferior, superior):
    return "confirmada" if inferior > 0 else "contrariada" if superior < 0 else "inconclusiva"


def pela_contagem(n, falhas):
    return "inconclusiva" if n == 0 else "confirmada" if falhas == 0 else "contrariada"


hipoteses = []
n = len(metros)


def registrar(hipotese, parte, estrategia, layout, n_pedidos, medida, valor, resultado,
              inferior=np.nan, superior=np.nan):
    hipoteses.append({"hipotese": hipotese, "parte": parte, "estrategia": estrategia, "layout": layout,
                      "n_pedidos": n_pedidos, "medida": medida, "valor": valor,
                      "ic_inf": inferior, "ic_sup": superior, "resultado": resultado})

### H1 e H2: redução entre layouts

In [14]:
for hipotese, comparacao in [("H1", "L1 vs L0"), ("H1", "L2 vs L0"), ("H2", "L1 vs L2")]:
    vistos = set()
    for r in reducoes[reducoes["comparacao"] == comparacao].itertuples():
        resultado = pelo_intervalo(r.ic_inf_pct, r.ic_sup_pct)
        vistos.add(resultado)
        registrar(hipotese, comparacao, r.estrategia, "-", n, "reducao_pct", r.reducao_pct, resultado,
                  r.ic_inf_pct, r.ic_sup_pct)
    resumo = vistos.pop() if len(vistos) == 1 else "mista"
    registrar(hipotese, comparacao, "as 4", "-", n, "resumo", np.nan, resumo)

### H3: ganho contra o carrinho, custoMin nunca perde e ganho crescendo com as faixas de seções

In [15]:
for r in ganhos.itertuples():
    registrar("H3", f"ganho contra {CARRINHO} > 0", r.estrategia, r.layout, n, "ganho_pct",
              r.ganho_medio_pct, pelo_intervalo(r.ic_inf_pct, r.ic_sup_pct), r.ic_inf_pct, r.ic_sup_pct)

for layout in LAYOUTS:
    melhor_outra = np.min([metros[layout, e] for e in ESTRATEGIAS if e != "custoMin"], axis=0)
    perdeu = metros[layout, "custoMin"].to_numpy() > melhor_outra + EMPATE
    for nome, grupo in [("pedidos exatos", ~heuristico), ("pedidos heuristicos", heuristico)]:
        total, falhas = int(grupo.sum()), int((perdeu & grupo).sum())
        registrar("H3", f"custoMin <= demais ({nome})", "custoMin", layout, total, "pedidos_violando",
                  falhas, pela_contagem(total, falhas))

for layout, estrategia in product(LAYOUTS, COMPARADAS):
    serie = faixas[(faixas["layout"] == layout) & (faixas["estrategia"] == estrategia)]
    quedas = int((serie["ganho_medio_pct"].diff() < 0).sum())
    registrar("H3", "ganho cresce com |S(o)| (faixas)", estrategia, layout, int(serie["n_pedidos"].sum()),
              "quedas_entre_faixas", quedas, "confirmada" if quedas == 0 else "inconclusiva")

### H4: pedidos de uma seção

In [16]:
uma_secao = secoes["L0"].to_numpy() == 1
for layout in LAYOUTS:
    todas = np.array([metros[layout, e] for e in ESTRATEGIAS])
    diferentes = todas.max(axis=0) - todas.min(axis=0) > EMPATE
    total, falhas = int(uma_secao.sum()), int((diferentes & uma_secao).sum())
    registrar("H4", "1 secao: as 4 estrategias iguais", "as 4", layout, total, "pedidos_diferentes",
              falhas, pela_contagem(total, falhas))

hipoteses = salvar(pd.DataFrame(hipoteses), "verificacao_hipoteses")

### Resumo

In [17]:
resumo = hipoteses[hipoteses["estrategia"].isin(["as 4", "custoMin"])]
resumo[["hipotese", "parte", "estrategia", "layout", "valor", "resultado"]]

,hipotese,parte,estrategia,layout,valor,resultado
3,H1,L1 vs L0,custoMin,-,7.351334,confirmada
4,H1,L1 vs L0,as 4,-,NaN,mista
8,H1,L2 vs L0,custoMin,-,23.866924,confirmada
9,H1,L2 vs L0,as 4,-,NaN,confirmada
13,H2,L1 vs L2,custoMin,-,-21.693055,contrariada
14,H2,L1 vs L2,as 4,-,NaN,contrariada
17,H3,ganho contra ordem_carrinho > 0,custoMin,L0,40.658983,confirmada
20,H3,ganho contra ordem_carrinho > 0,custoMin,L1,45.784028,confirmada
23,H3,ganho contra ordem_carrinho > 0,custoMin,L2,43.591794,confirmada
24,H3,custoMin <= demais (pedidos exatos),custoMin,L0,0.000000,confirmada
